In [20]:
def crop(Image, offsetHauteur, offsetLargeur):
    hauteur = Image.shape[0]
    largeur = Image.shape[1]
    croped_image = Image[(hauteur//2 -offsetHauteur):(hauteur//2 +offsetHauteur),(largeur//2 -offsetLargeur):(largeur//2 +offsetLargeur)]
    return croped_image

In [21]:
import cv2 as cv
import numpy as np
import time
import math
pathVideo = "./video/rl4_pb8-7.mp4"
start = time.process_time()

# Parameters for Shi-Tomasi corner detection
feature_params = dict(maxCorners = 0, qualityLevel = 0.2, minDistance = 10, blockSize = 7)
# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize = (15,15), maxLevel = 2, criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))
# The video feed is read in as a VideoCapture object
cap = cv.VideoCapture(pathVideo)
if not cap.isOpened():
    print("Cannot open the file !")
# Variable for color to draw optical flow track
color = (0, 255, 0)
# pixel to mm
pixels_to_mm = 1
# get the FPS of the video
fps = int(cap.get(cv.CAP_PROP_FPS))
# set the step for taking frames based on the FPS
### step = int((1/fps) * 1000) # msec in this case the programme will be slower
step = 150 # this is the best value
# set the counter of frames
count = 0
# ret = a boolean return value from getting the frame, first_frame = the first frame in the entire video sequence
ret, first_frame = cap.read()
# crop the frame
first_frame = crop(first_frame,200,200)
# Converts frame to grayscale because we only need the luminance channel for detecting edges - less computationally expensive
prev_gray = cv.cvtColor(first_frame, cv.COLOR_BGR2GRAY)
# Otsu's thresholding
ret2,prev_gray = cv.threshold(prev_gray,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)
# Finds the strongest corners in the first frame by Shi-Tomasi method - we will track the optical flow for these corners
# https://docs.opencv.org/3.0-beta/modules/imgproc/doc/feature_detection.html#goodfeaturestotrack
prev = cv.goodFeaturesToTrack(prev_gray, mask = None, **feature_params)
# Creates an image filled with zero intensities with the same dimensions as the frame - for later drawing purposes
mask = np.zeros_like(first_frame)

while(True):
    # set the counter of frames
    count = count + 1
    # set the position to take the frame
    cap.set(cv.CAP_PROP_POS_MSEC,(count*step))
    # ret = a boolean return value from getting the frame, frame = the current frame being projected in the video
    ret, frame = cap.read()
    if ret == False:
        break
    # crop the frame
    frame = crop(frame,200,200)
    # Converts each frame to grayscale - we previously only converted the first frame to grayscale
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    # Otsu's thresholding
    ret2,gray = cv.threshold(gray,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)
    # Calculates sparse optical flow by Lucas-Kanade method
    # https://docs.opencv.org/3.0-beta/modules/video/doc/motion_analysis_and_object_tracking.html#calcopticalflowpyrlk
    prev = cv.goodFeaturesToTrack(prev_gray, mask = None, **feature_params)
    next, status, error = cv.calcOpticalFlowPyrLK(prev_gray, gray, prev, None, **lk_params)
    # Selects good feature points for previous position
    good_old = prev[status == 1].astype(int)
    # Selects good feature points for next position
    good_new = next[status == 1].astype(int)
    # Creates an image filled with zero intensities with the same dimensions as the frame - for later drawing purposes
    mask = np.zeros_like(first_frame)
    # Draws the optical flow tracks
    vx_list = []
    vy_list = []
    dt = step/1000 # dt in second fixed for all frames
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        # Returns a contiguous flattened array as (x, y) coordinates for new point
        x_new, y_new = new.ravel()
        # Returns a contiguous flattened array as (x, y) coordinates for old point
        x_old, y_old = old.ravel()
        # calculate the speed in x and y axis
        vx = abs((x_new-x_old)*pixels_to_mm/dt)
        vy = abs((y_new-y_old)*pixels_to_mm/dt)
        vx_list.append(vx)
        vy_list.append(vy)
        # Draws line between new and old position with green color and 2 thickness
        mask = cv.line(mask, (x_new, y_new), (x_old, y_old), color, 2)
        # Draws filled circle (thickness of -1) at new position with green color and radius of 3
        frame = cv.circle(frame, (x_new, y_new), 3, (0, 0, 255), -1)
    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)
    # Overlays the optical flow tracks on the original frame
    output = cv.add(frame, mask)
    # Updates previous frame
    prev_gray = gray.copy()
    # resize the image 
    ####scale_percent = 50 # percent of original size
    ####width = int(output.shape[1] * scale_percent / 100)
    ####height = int(output.shape[0] * scale_percent / 100)
    ####dim = (width, height)
    ####output = cv.resize(output, dim, interpolation = cv.INTER_AREA)
    ####frame = cv.resize(frame, dim, interpolation = cv.INTER_AREA)
    ####gray = cv.resize(gray, dim, interpolation = cv.INTER_AREA)
    # image to display speed measures
    speed = np.zeros((250, 600), dtype = np.uint8)
    # whritw the speed result in the video
    text_vx = "vx = " + str(average_vx)
    text_vy = "vy = " + str(average_vy)
    text_v = "v = " + str(math.sqrt(pow(average_vx,2) + pow(average_vy,2)))
    font = cv.FONT_HERSHEY_SIMPLEX
    fontScale = 1
    cv.putText(speed, text_vx, (50, 50), font, fontScale, (255, 0, 0), 3)
    cv.putText(speed, text_vx, (50, 50), font, fontScale, (255, 0, 0), 3)
    
    cv.putText(speed, text_vy, (50, 100), font, fontScale, (255, 0, 0), 3)
    cv.putText(speed, text_vy, (50, 100), font, fontScale, (255, 0, 0), 3)
    cv.putText(speed, text_v, (50, 150), font, fontScale, (255, 0, 0), 3)
    # Opens a new window and displays the output frame
    cv.imshow("speed measures", speed)
    cv.imshow("sparse optical flow", output)
    cv.imshow("frame", frame)
    cv.imshow("Otsu's thresholding", gray)
    # Frames are read by intervals of 10 milliseconds. The programs breaks out of the while loop when the user presses the 'q' key
    if cv.waitKey(1) == ord('q'):
        break
print("==> optical flow ",time.process_time() - start)
# The following frees up resources and closes all windows
cap.release()
cv.destroyAllWindows()

==> optical flow  133.78125


In [19]:
import random
color = tuple(random.randint(0,255) for _ in range(3))
print(color)

(160, 170, 30)
